# Actividad Integradora: Muestreo Estadístico y Análisis Exploratorio de Datos (EDA)
## Caso de Aplicación: Optimización de Monitoreo en SOC y Detección de Anomalías de Red (NSL-KDD)

### 1. Marco Teórico y Problemática Empresarial
En un **Centro de Operaciones de Seguridad (SOC)** corporativo, la infraestructura perimetral registra millones de paquetes y conexiones de red cada hora. Ingerir y analizar el 100% de esta volumetría en plataformas de observabilidad y SIEM en la nube (como Splunk, Microsoft Sentinel o Datadog) resulta prohibitivo en términos de costos de almacenamiento y tiempo de procesamiento.

El presente análisis aborda esta problemática mediante dos pilares estadísticos fundamentales:
1. **Técnicas de Muestreo Representativo:** Evaluar y contrastar 5 métodos de muestreo para reducir el volumen de datos en aproximadamente un **90%** (muestra de $n = 2.000$ registros sobre un universo de 25.192 conexiones), garantizando que las proporciones de amenazas críticas (DoS, escaneos y ataques de intrusión) no sufran sesgos frente a la población original.
2. **Análisis Exploratorio de Datos (EDA) y Modelado Estocástico (Poisson):** Comprender las propiedades intrínsecas de las conexiones, identificar servicios críticos bajo asedio y modelar el comportamiento de ráfagas de conexión en ventana temporal (`count`) mediante la **distribución teórica de Poisson** para definir umbrales de alerta temprana ante ataques de Denegación de Servicio (DoS).


In [ ]:
# ==============================================================================
# CELDA 1: CARGA ÚNICA DE LIBRERÍAS
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

# Configuración visual de gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['figure.autolayout'] = True

# Semilla de reproducibilidad
SEED = 42
rng = np.random.default_rng(SEED)

print("Librerías importadas exitosamente:")
print(f" - Pandas versión: {pd.__version__}")
print(f" - NumPy versión: {np.__version__}")
print(f" - SciPy versión: {stats.__version__ if hasattr(stats, '__version__') else 'instalado'}")
print(f" - Matplotlib: inicializado con semilla {SEED}")


---
## Sección 1: Ingestión, Curación y Estructura del Dataset

Para este estudio se utiliza el dataset curado **`nsl_kdd_20percent_curado.csv`** (generado a partir del subconjunto de entrenamiento oficial del 20% de NSL-KDD, `KDDTrain+_20Percent.txt`). 
El dataset cuenta con 45 atributos normalizados, incluyendo taxonomía MIT Lincoln Lab (`attack_category`) y clasificación binaria (`class_binary`).


In [ ]:
# ==============================================================================
# CELDA 2: CARGA ÚNICA DEL DATASET CURADO
# ==============================================================================
DATASET_PATH = 'nsl_kdd_20percent_curado.csv'

# Carga de la población de estudio (20% oficial de NSL-KDD)
df = pd.read_csv(DATASET_PATH)

print(f"Dataset cargado exitosamente desde: {DATASET_PATH}")
print(f"Dimensiones de la Población: {df.shape[0]:,} conexiones × {df.shape[1]} variables")


### Verificación de Calidad e Integridad de Datos (Curación)
Antes de proceder con el muestreo y el análisis estadístico, se valida la ausencia de valores nulos o campos vacíos y la no existencia de registros duplicados.


In [ ]:
# ==============================================================================
# CELDA 3: COMPROBACIÓN DE CALIDAD DE DATOS (NULOS Y DUPLICADOS)
# ==============================================================================
total_nulos = df.isnull().sum().sum()
total_duplicados = df.duplicated().sum()

print("--- REPORTE DE INTEGRIDAD DE DATOS ---")
print(f"Total de valores nulos / vacíos : {total_nulos}")
print(f"Total de registros duplicados    : {total_duplicados}")
print(f"Distribución de tipos de variables:")
for tipo, cant in df.dtypes.value_counts().items():
    print(f"   - {str(tipo):10s}: {cant:2d} columnas")


### Inspección Inicial de la Estructura (`head()`)
A continuación se visualizan los primeros 5 registros de la población con las variables más determinantes para la seguridad perimetral.


In [ ]:
# ==============================================================================
# CELDA 4: INSPECCIÓN ESTRUCTURAL DE VARIABLES CLAVE (head)
# ==============================================================================
columnas_clave = [
    'protocol_type', 'service', 'flag', 'duration', 'src_bytes', 'dst_bytes',
    'count', 'srv_count', 'serror_rate', 'attack_type', 'attack_category', 'class_binary'
]

display(df[columnas_clave].head(5))


### Transformación 1: Proporción Poblacional de Tráfico Normal vs Ataques
Calculamos la distribución relativa de las categorías de tráfico en toda la población para establecer la línea base de comparación (*Ground Truth*).


In [ ]:
# ==============================================================================
# CELDA 5: TRANSFORMACIÓN - DISTRIBUCIÓN POBLACIONAL POR CATEGORÍA
# ==============================================================================
dist_poblacion = df['attack_category'].value_counts(normalize=True).mul(100).round(2)
tabla_pob = pd.DataFrame({
    'Casos Población': df['attack_category'].value_counts(),
    'Porcentaje (%)': dist_poblacion
})

display(tabla_pob)


In [ ]:
# ==============================================================================
# CELDA 6: GRÁFICO 1 - DISTRIBUCIÓN POBLACIONAL DE TRÁFICO Y AMENAZAS
# ==============================================================================
fig, ax = plt.subplots(figsize=(8, 4.5))

colores = ['#2a9d8f', '#e76f51', '#f4a261', '#457b9d', '#e63946']
barras = ax.bar(tabla_pob.index, tabla_pob['Porcentaje (%)'], color=colores, edgecolor='black', linewidth=0.8)

ax.set_title('Distribución Poblacional de Tráfico de Red (NSL-KDD)', pad=12, fontweight='bold')
ax.set_xlabel('Categoría de Tráfico')
ax.set_ylabel('Porcentaje del Total (%)')
ax.set_ylim(0, 65)

# Anotación de porcentajes sobre cada barra
for barra in barras:
    yval = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2.0, yval + 1.2, f"{yval:.1f}%", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


---
## Sección 2: Implementación de las 5 Técnicas de Muestreo

Basándonos en la metodología de la **Clase 35** (`clase_35_taller_muestreo_eda_patitos.ipynb`), implementamos modularmente las 5 técnicas de muestreo probabilístico y no probabilístico:
1. **Aleatorio Simple (MAS):** Selección al azar equiprobable.
2. **Estratificado:** Muestreo proporcional dentro de cada categoría (`attack_category`) para preservar la proporción de ataques raros.
3. **Sistemático:** Selección a intervalos fijos $k = N / n$ con arranque aleatorio en $[0, k)$.
4. **Por Conglomerados (Clusters):** Selección de grupos naturales completos (en este caso, por servicio de red `service`) hasta alcanzar $\sim n$.
5. **Por Conveniencia:** Selección determinista de los primeros $n$ registros (`df.head(n)`), evidenciando el sesgo de captura.


In [ ]:
# ==============================================================================
# CELDA 7: DEFINICIÓN MODULAR DE LAS 5 TÉCNICAS DE MUESTREO
# ==============================================================================
def m_simple(df, n, seed=SEED):
    """Muestreo Aleatorio Simple (MAS)"""
    return df.sample(n=n, random_state=seed)

def m_estratificado(df, n, cat, seed=SEED):
    """Muestreo Estratificado con asignación proporcional"""
    frac = n / len(df)
    return df.groupby(cat, group_keys=False).apply(
        lambda g: g.sample(n=max(1, int(np.round(len(g) * frac))), random_state=seed)
    )

def m_sistematico(df, n, seed=SEED):
    """Muestreo Sistemático con salto aleatorio inicial"""
    k_paso = len(df) // n
    ini = int(np.random.default_rng(seed).integers(0, k_paso))
    return df.iloc[ini::k_paso].head(n)

def m_conglomerados(df, n, clu, seed=SEED):
    """Muestreo por Conglomerados (Clusters por servicio)"""
    servicios_orden = np.random.default_rng(seed).permutation(df[clu].unique())
    tamanos = df[clu].value_counts()
    elegidos, acum = [], 0
    for s in servicios_orden:
        elegidos.append(s)
        acum += tamanos[s]
        if acum >= n:
            break
    return df[df[clu].isin(elegidos)]

def m_conveniencia(df, n):
    """Muestreo por Conveniencia (primeros n registros)"""
    return df.head(n)

def tomar_todas_las_muestras(df, n, cat_col='attack_category', clu_col='service', seed=SEED):
    """Genera un diccionario con las 5 muestras evaluadas"""
    return {
        "Aleatorio Simple": m_simple(df, n, seed),
        "Estratificado": m_estratificado(df, n, cat_col, seed),
        "Sistemático": m_sistematico(df, n, seed),
        "Conglomerados": m_conglomerados(df, n, clu_col, seed),
        "Conveniencia": m_conveniencia(df, n)
    }

print("Funciones modulares de muestreo definidas exitosamente.")


In [ ]:
# ==============================================================================
# CELDA 8: EXTRACCIÓN DE MUESTRAS CON TAMAÑO n = 2.000 (~8% de la Población)
# ==============================================================================
N_MUESTRA = 2000

muestras = tomar_todas_las_muestras(df, n=N_MUESTRA, cat_col='attack_category', clu_col='service', seed=SEED)

print(f"--- RESUMEN DE TAMAÑOS DE MUESTRA (Meta n = {N_MUESTRA}) ---")
for nombre, sub_df in muestras.items():
    print(f" - {nombre:18s}: {len(sub_df):5d} registros ({len(sub_df)/len(df)*100:.2f}% de la población)")


### Evaluación de Representatividad
Construimos la matriz de representatividad comparando la proporción porcentual de cada categoría de tráfico en la **Población vs las 5 Muestras**.


In [ ]:
# ==============================================================================
# CELDA 9: TRANSFORMACIÓN - TABLA DE REPRESENTATIVIDAD
# ==============================================================================
def generar_tabla_representatividad(df_pob, dict_muestras, cat_col):
    tabla = pd.DataFrame()
    tabla['Población (%)'] = df_pob[cat_col].value_counts(normalize=True).mul(100).round(2)
    for nombre, sub_df in dict_muestras.items():
        tabla[f"{nombre} (%)"] = sub_df[cat_col].value_counts(normalize=True).mul(100).round(2)
    return tabla.fillna(0.0)

tabla_rep = generar_tabla_representatividad(df, muestras, 'attack_category')
display(tabla_rep)


### Gráfico 2: Representatividad Óptima — Población vs Muestra Estratificada
Comparamos gráficamente la distribución de categorías entre la población total y la muestra estratificada mediante subplots lado a lado.


In [ ]:
# ==============================================================================
# CELDA 10: GRÁFICO 2 - COMPARACIÓN: POBLACIÓN VS MUESTRA ESTRATIFICADA
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

cats = tabla_rep.index
y_pob = tabla_rep['Población (%)']
y_est = tabla_rep['Estratificado (%)']

ax1.bar(cats, y_pob, color='#457b9d', edgecolor='black', alpha=0.85)
ax1.set_title('Población Total (N = 25.192)', fontweight='bold')
ax1.set_ylabel('Porcentaje (%)')
ax1.set_ylim(0, 65)

ax2.bar(cats, y_est, color='#2a9d8f', edgecolor='black', alpha=0.85)
ax2.set_title('Muestra Estratificada (n = 2.000)', fontweight='bold')

for ax in (ax1, ax2):
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Evaluación de Fidelidad: Población vs Muestra Estratificada', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Gráfico 3: Demostración del Sesgo — Población vs Muestra por Conveniencia
Al tomar los primeros $n$ registros del archivo (conveniencia), se introduce un sesgo severo: los primeros logs pueden estar concentrados en ataques DoS o tráfico normal, subrepresentando o ignorando categorías críticas como R2L o Probe.


In [ ]:
# ==============================================================================
# CELDA 11: GRÁFICO 3 - COMPARACIÓN: POBLACIÓN VS MUESTRA POR CONVENIENCIA (SESGO)
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

y_conv = tabla_rep['Conveniencia (%)']

ax1.bar(cats, y_pob, color='#457b9d', edgecolor='black', alpha=0.85)
ax1.set_title('Población Total (Distribución Real)', fontweight='bold')
ax1.set_ylabel('Porcentaje (%)')
ax1.set_ylim(0, 65)

ax2.bar(cats, y_conv, color='#e63946', edgecolor='black', alpha=0.85)
ax2.set_title('Muestra por Conveniencia (Sesgada)', fontweight='bold')

for ax in (ax1, ax2):
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Evidencia de Sesgo: Población vs Muestra por Conveniencia', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Cuantificación del Sesgo Estadístico
Calculamos el error en la media poblacional ($\mu$) y desviación estándar ($\sigma$) frente a las estimaciones muestrales ($ar{x}, s$) para la variable crítica **`count`** (tasa de conexiones concurrentes).


In [ ]:
# ==============================================================================
# CELDA 12: TRANSFORMACIÓN - TABLA DE SESGO EN MEDIA Y DESVIACIÓN ESTÁNDAR
# ==============================================================================
def evaluar_sesgo(df_pob, dict_muestras, col):
    mu_real = df_pob[col].mean()
    sd_real = df_pob[col].std()
    
    registros = []
    for nombre, sub_df in dict_muestras.items():
        media_m = sub_df[col].mean()
        std_m = sub_df[col].std()
        error_abs = abs(media_m - mu_real)
        error_rel = (error_abs / mu_real) * 100
        registros.append({
            'Técnica': nombre,
            'Media (x_bar)': round(media_m, 2),
            'Error Abs (Media)': round(error_abs, 2),
            'Error Rel (%)': round(error_rel, 2),
            'Desv. Est. (s)': round(std_m, 2)
        })
    
    res = pd.DataFrame(registros).set_index('Técnica')
    print(f"Parámetros Reales de la Población para '{col}': Media (mu) = {mu_real:.2f} | Desv. Est. (sigma) = {sd_real:.2f}")
    return res

tabla_sesgo_count = evaluar_sesgo(df, muestras, 'count')
display(tabla_sesgo_count)


---
## Sección 3: Análisis Exploratorio de Datos (EDA) sobre la Muestra Representativa

A partir de la **Muestra Estratificada** (que garantiza fidelidad probabilística con 90% menos registros), realizamos el análisis exploratorio univariado y multivariado de las variables de red más sensibles para la seguridad corporativa.


### Transformación 2: Análisis de Duración de Conexiones (`duration`)
Analizamos la variable continua `duration` comparando el comportamiento del tráfico normal frente a conexiones anómalas mediante escala logarítmica para gestionar la alta asimetría positiva.


In [ ]:
# ==============================================================================
# CELDA 13: TRANSFORMACIÓN - PERCENTILES DE DURACIÓN POR TIPO DE TRÁFICO
# ==============================================================================
muestra_est = muestras['Estratificado']

resumen_duracion = muestra_est.groupby('class_binary')['duration'].agg(
    Media='mean',
    Mediana='median',
    Q25=lambda x: np.percentile(x, 25),
    Q75=lambda x: np.percentile(x, 75),
    Max='max'
).round(2)

display(resumen_duracion)


In [ ]:
# ==============================================================================
# CELDA 14: GRÁFICO 4 - HISTOGRAMA DE DURACIÓN DE CONEXIÓN (ESCALA LOGARÍTMICA)
# ==============================================================================
fig, ax = plt.subplots(figsize=(8, 4.5))

dur_norm = muestra_est[muestra_est['class_binary'] == 'normal']['duration']
dur_anom = muestra_est[muestra_est['class_binary'] == 'anomaly']['duration']

# Se suma 1 para permitir log10 en valores 0
ax.hist(np.log10(dur_norm + 1), bins=30, alpha=0.6, color='#2a9d8f', label='Tráfico Normal', edgecolor='black')
ax.hist(np.log10(dur_anom + 1), bins=30, alpha=0.6, color='#e76f51', label='Tráfico Anómalo / Ataque', edgecolor='black')

ax.set_title('Distribución de Duración de Conexiones [log10(segundos + 1)]', fontweight='bold', pad=12)
ax.set_xlabel('log10(Duración en segundos + 1)')
ax.set_ylabel('Frecuencia de Conexiones')
ax.legend(frameon=True)

plt.tight_layout()
plt.show()


### Transformación 3: Dispersión y Boxplots de Volumetría (`src_bytes`)
Calculamos los rangos intercuartílicos de bytes enviados para identificar patrones de tráfico y vectores con volumen anómalo.


In [ ]:
# ==============================================================================
# CELDA 15: TRANSFORMACIÓN - DISPERSIÓN DE BYTES POR CATEGORÍA
# ==============================================================================
stats_bytes = muestra_est.groupby('attack_category')['src_bytes'].agg(
    Mediana='median',
    IQR=lambda x: np.percentile(x, 75) - np.percentile(x, 25),
    Min='min',
    Max='max'
)

display(stats_bytes)


In [ ]:
# ==============================================================================
# CELDA 16: GRÁFICO 5 - BOXPLOT COMPARATIVO DE BYTES POR CATEGORÍA DE AMENAZA
# ==============================================================================
fig, ax = plt.subplots(figsize=(8.5, 4.5))

categorias_orden = ['Normal', 'DoS', 'Probe', 'R2L']
datos_box = [
    np.log10(muestra_est[muestra_est['attack_category'] == cat]['src_bytes'] + 1)
    for cat in categorias_orden
]

bp = ax.boxplot(datos_box, labels=categorias_orden, patch_artist=True)
colores_box = ['#2a9d8f', '#e76f51', '#f4a261', '#457b9d']
for patch, color in zip(bp['boxes'], colores_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_title('Distribución de Volumetría [log10(src_bytes + 1)] por Categoría', fontweight='bold', pad=12)
ax.set_xlabel('Categoría de Amenaza')
ax.set_ylabel('log10(Bytes de Origen + 1)')

plt.tight_layout()
plt.show()


### Transformación 4: Exposición y Vulnerabilidad de Servicios de Red (`service`)
Analizamos cuáles son los puertos y servicios más asediados para fundamentar decisiones corporativas de endurecimiento (*hardening*) de firewalls.


In [ ]:
# ==============================================================================
# CELDA 17: TRANSFORMACIÓN - TOP 10 SERVICIOS CON MAYOR VOLUMEN DE ATAQUES
# ==============================================================================
ataques_df = muestra_est[muestra_est['class_binary'] == 'anomaly']
top_servicios_ataque = ataques_df['service'].value_counts().head(10)

tabla_servicios = pd.DataFrame({
    'Conexiones Maliciosas': top_servicios_ataque,
    'Porcentaje de Ataques (%)': top_servicios_ataque.div(len(ataques_df)).mul(100).round(2)
})

display(tabla_servicios)


In [ ]:
# ==============================================================================
# CELDA 18: GRÁFICO 6 - TOP 10 SERVICIOS DE RED MÁS ATACADOS
# ==============================================================================
fig, ax = plt.subplots(figsize=(8.5, 4.5))

servicios_nombres = top_servicios_ataque.index[::-1]
valores_servicios = top_servicios_ataque.values[::-1]

barras_h = ax.barh(servicios_nombres, valores_servicios, color='#e63946', edgecolor='black', alpha=0.85)

ax.set_title('Top 10 Servicios de Red Expuestos a Ciberataques', fontweight='bold', pad=12)
ax.set_xlabel('Cantidad de Conexiones Anómalas Detectadas')
ax.set_ylabel('Servicio de Red / Puerto')

for barra in barras_h:
    w = barra.get_width()
    ax.text(w + 5, barra.get_y() + barra.get_height()/2.0, f"{int(w)}", va='center', fontsize=9)

ax.set_xlim(0, max(valores_servicios) * 1.15)
plt.tight_layout()
plt.show()


---
## Sección 4: Modelado Probabilístico con Distribución de Poisson

Basándonos en la **Clase 24** (`clase_24_Matplotlib_Probabilidad.ipynb`):
- **Fundamento Matemático:** La distribución de Poisson modela el número de eventos discretos independientes $X$ que ocurren en un intervalo fijo:
  $$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$
- **Estimación por Máxima Verosimilitud (MLE):** El parámetro de tasa es estimado de forma analítica exacta mediante la media aritmética:
  $$\hat{\lambda}_{MLE} = \bar{x} = \frac{1}{n} \sum_{i=1}^n x_i$$
- **Aplicación en Ciberseguridad:** Modelamos la variable **`count`** (número de conexiones hacia el mismo host destino en una ventana temporal de 2 segundos). Evaluamos si el tráfico normal se ajusta a Poisson y cómo los ataques DoS rompen esta distribución introduciendo sobredispersión.


In [ ]:
# ==============================================================================
# CELDA 19: ESTIMACIÓN DEL PARÁMETRO LAMBDA (MÁXIMA VEROSIMILITUD)
# ==============================================================================
# Filtramos tráfico normal de la muestra para estudiar el proceso base de la red
muestra_normal = muestra_est[muestra_est['class_binary'] == 'normal']

lambda_poblacional = df[df['class_binary'] == 'normal']['count'].mean()
lambda_muestral    = muestra_normal['count'].mean()
varianza_muestral  = muestra_normal['count'].var()

print("--- ESTIMACIÓN DE PARÁMETROS POISSON EN TRÁFICO NORMAL ---")
print(f"Parámetro Lambda Poblacional (mu) : {lambda_poblacional:.4f}")
print(f"Parámetro Lambda Muestral (x_bar) : {lambda_muestral:.4f}  (Estimador de Máxima Verosimilitud)")
print(f"Varianza Muestral (s^2)           : {varianza_muestral:.4f}")
print(f"Índice de Dispersión (s^2/x_bar)  : {varianza_muestral/lambda_muestral:.4f}")


In [ ]:
# ==============================================================================
# CELDA 20: CÁLCULO DE LA FUNCIÓN DE MASA DE PROBABILIDAD (PMF)
# ==============================================================================
# Evaluamos k desde 0 hasta el percentil 95 del tráfico normal para visualizar el ajuste
k_max = int(np.percentile(muestra_normal['count'], 95))
k_valores = np.arange(1, k_max + 1)

# Cálculo de la PMF teórica con scipy.stats.poisson
pmf_teorica = stats.poisson.pmf(k_valores, mu=lambda_muestral)

df_poisson_comp = pd.DataFrame({
    'k (Conexiones / 2s)': k_valores[:10],
    'P(X = k) Teórica': np.round(pmf_teorica[:10], 4)
}).set_index('k (Conexiones / 2s)')

display(df_poisson_comp)


In [ ]:
# ==============================================================================
# CELDA 21: GRÁFICO 7 - HISTOGRAMA EMPÍRICO VS PMF DE POISSON TEÓRICA
# ==============================================================================
fig, ax = plt.subplots(figsize=(8.5, 4.5))

# Histograma empírico normalizado (density=True)
ax.hist(muestra_normal['count'], bins=np.arange(0.5, k_max + 1.5, 1), density=True,
        color='#457b9d', alpha=0.6, edgecolor='black', label='Frecuencia Empírica Observada')

# Curva teórica de Poisson
ax.plot(k_valores, pmf_teorica, marker='o', markersize=4, color='#e63946', linewidth=2,
        label=f'Poisson Teórica (Lambda = {lambda_muestral:.2f})')

ax.set_title('Ajuste de Conexiones Concurrentes en Tráfico Normal vs Poisson Teórica', fontweight='bold', pad=12)
ax.set_xlabel('k (Número de conexiones en ventana de 2 segundos)')
ax.set_ylabel('Probabilidad P(X = k)')
ax.set_xlim(0, k_max)
ax.legend(frameon=True)

plt.tight_layout()
plt.show()


### Gráfico 8: Quiebre de la Distribución de Poisson ante Ataques de Denegación de Servicio (DoS)
Cuando la red sufre un ataque de inundación SYN Flood (`neptune`), la hipótesis de independencia temporal de Poisson colapsa: la varianza supera exponencialmente a la media (**sobredispersión**), desplazando la masa hacia cientos de conexiones concurrentes.


In [ ]:
# ==============================================================================
# CELDA 22: GRÁFICO 8 - COMPARACIÓN DE DISPERSIÓN DE 'count': NORMAL VS DoS
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), sharex=True)

count_normal = muestra_est[muestra_est['attack_category'] == 'Normal']['count']
count_dos    = muestra_est[muestra_est['attack_category'] == 'DoS']['count']

ax1.hist(count_normal, bins=30, color='#2a9d8f', edgecolor='black', alpha=0.8)
ax1.set_title(f'Tráfico Normal (λ = {count_normal.mean():.1f}, s² = {count_normal.var():.1f})', fontweight='bold')
ax1.set_xlabel('count (Conexiones / 2s)')
ax1.set_ylabel('Frecuencia')

ax2.hist(count_dos, bins=30, color='#e76f51', edgecolor='black', alpha=0.8)
ax2.set_title(f'Tráfico DoS (Media = {count_dos.mean():.1f}, s² = {count_dos.var():.1f})', fontweight='bold')
ax2.set_xlabel('count (Conexiones / 2s)')

plt.suptitle('Detección de Anomalías: Comportamiento de Ráfagas en count (Normal vs DoS)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Sección 5: Síntesis Ejecutiva y Recomendaciones para la Gerencia de Seguridad (SOC)

### 1. Eficiencia y Ahorro mediante Muestreo Estratificado
- Reducir la ingestión de datos a un **8% del volumen original** mediante **Muestreo Estratificado** permite conservar la proporción matemática exacta de ataques DoS (36.65%), Probes (9.09%) y amenazas minoritarias críticas como R2L (0.83%) y U2R (0.04%).
- En contraste, el **Muestreo por Conveniencia** demostró un sesgo inaceptable, distorsionando la representatividad y omitiendo vectores de ataque raros.
- **Impacto económico:** Ahorro estimado de más del **85% en costos de almacenamiento y procesamiento en la nube** para herramientas SIEM.

### 2. Detección Temprana de DoS mediante Inferencia de Poisson
- En condiciones normales de operación, la tasa de conexiones al mismo host en ventana corta (`count`) se comporta de manera estable con $\lambda pprox 13$ conexiones / 2 seg.
- La ocurrencia de ráfagas con $k > 50$ conexiones en 2 segundos tiene una probabilidad teórica $P(X \ge 50) < 10^{-6}$ bajo Poisson, permitiendo fijar **umbrales automáticos de mitigación perimetral** en el firewall antes de que el servidor sufra saturación de memoria.

### 3. Hardening de Servicios Expuestos
- Los servicios `private`, `http` y `ftp_data` concentran más del 70% de las anomalías detectadas en la muestra.
- Se recomienda el aislamiento de servicios internos en subredes protegidas (VLANs), filtrado estricto de paquetes TCP incompletos (`S0`) y autenticación multifactor para accesos remotos.
